In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Got the data

In [ ]:
df = pd.read_csv("./dataset1.csv")
print(df.head())

   feature1  feature2  feature3  observation
0  0.496714 -0.138264  0.647689     5.642777
1  1.523030 -0.234153 -0.234137    -1.768262
2  1.579213  0.767435 -0.469474     0.197153
3  0.542560 -0.463418 -0.465730     0.727206
4  0.241962 -1.913280 -1.724918    -8.197938


In [4]:
features = df.iloc[:, :3].to_numpy()
Y = df.iloc[:, 3].to_numpy()

train_split = int(0.8 * len(features))

features_train = features[:train_split]
y_train = Y[:train_split]

features_test = features[train_split:]
y_test = Y[train_split:]

# Solve for the Coefficients

In [5]:
A = np.column_stack((features_train, np.ones(len(features_train))))
print(A.shape)
A[:5]

(80, 4)


array([[ 0.49671415, -0.1382643 ,  0.64768854,  1.        ],
       [ 1.52302986, -0.23415337, -0.23413696,  1.        ],
       [ 1.57921282,  0.76743473, -0.46947439,  1.        ],
       [ 0.54256004, -0.46341769, -0.46572975,  1.        ],
       [ 0.24196227, -1.91328024, -1.72491783,  1.        ]])

In [6]:
coefficients_hat = np.linalg.inv(A.T @ A) @ A.T @ y_train
print(coefficients_hat)

[-3.09489358  1.97061292  4.87896787  5.14581759]


# Display

In [23]:
fig = go.Figure(
    data=go.Scatter3d(
        x=features_train[:, 0],
        y=features_train[:, 1],
        z=features_train[:, 2],
        mode="markers",
        marker=dict(
            size=6,
            color=Y,
            colorscale="Inferno",
            opacity=0.8,
            showscale=True,
        ),
    )
)
fig.show()

In [9]:
x_min, x_max = features[:, 0].min(), features[:, 0].max()
y_min, y_max = features[:, 1].min(), features[:, 1].max()
z_min, z_max = features[:, 2].min(), features[:, 2].max()

x, y, z = np.mgrid[x_min:x_max:30j, y_min:y_max:30j, z_min:z_max:30j]
a, b, c, d = coefficients_hat

values = a * x + b * y + c * z + d

fig = go.Figure(
    data=go.Volume(
        x=x.flatten(),
        y=y.flatten(),
        z=z.flatten(),
        value=values.flatten(),
        isomin=values.min(),
        isomax=values.max(),
        opacity=0.7,
        surface_count=20,
        colorscale="Inferno",
    )
)

fig.show()

In [21]:
fig = go.Figure()

fig.add_trace(
    go.Volume(
        x=x.flatten(),
        y=y.flatten(),
        z=z.flatten(),
        value=values.flatten(),
        isomin=values.min(),
        isomax=values.max(),
        opacity=0.7,
        surface_count=20,
        colorscale="Inferno",
    )
)


fig.add_trace(
    go.Scatter3d(
        x=features_train[:, 0],
        y=features_train[:, 1],
        z=features_train[:, 2],
        mode="markers",
        marker=dict(
            size=6,
            color=Y,
            colorscale="Inferno",
            opacity=0.8,
            showscale=True,
        ),
    )
)

fig.show()

# Testing bada ba pa pa eat more chikn

In [ ]:
A_test = np.column_stack((features_test, np.ones(len(features_test))))
Y_pred = A_test @ coefficients_hat

residuals = y_test - Y_pred # Residual or diff

mse = np.mean(residuals**2)
rmse = np.sqrt(mse)

ss_residual = np.sum(residuals**2)
ss_total = np.sum((y_test - np.mean(y_test)) ** 2)
r2_score = 1 - (ss_residual / ss_total)

print(f"R^2: {r2_score:.4f}") # Variance
print(f"RMSE: {rmse:.4f}") # Basically misses by 1.1 on average

R^2: 0.9613
RMSE: 1.1407


In [22]:
fig = go.Figure()

fig.add_trace(
    go.Volume(
        x=x.flatten(),
        y=y.flatten(),
        z=z.flatten(),
        value=values.flatten(),
        isomin=values.min(),
        isomax=values.max(),
        opacity=0.7,
        surface_count=20,
        colorscale="Inferno",
    )
)


fig.add_trace(
    go.Scatter3d(
        x=features_test[:, 0],
        y=features_test[:, 1],
        z=features_test[:, 2],
        mode="markers",
        marker=dict(
            size=6,
            color=Y,
            colorscale="Inferno",
            opacity=0.8,
            showscale=True,
        ),
    )
)

fig.show()